# 100게임 카드별 등장 확률 검증

VS Code에서 이 노트북을 열고 셀을 위에서 아래로 **Ctrl+Enter** 실행합니다.

검증 규칙:
- 표준 52장을 매 게임 균등 셔플하며 한 번 나온 카드는 다시 나오지 않습니다.
- 펼치기 1회에 플레이어와 AI가 카드 1장씩 공개합니다.
- 두 카드 공개 후 좌우 해골 합계를 판정합니다.
- 좌우 각 더미는 최신 카드 최대 2장만 노출·합산하며, 선택 더미의 1장·2장 획득을 구분합니다.
- 벨 승부 후 반대편 필드 카드도 미획득 카드로 제거합니다.
- 한쪽이 벨 3승을 달성하면 게임을 종료합니다.
- 해골 공식 배정표가 없으므로 1·2·3을 18·17·17장으로 고정 배정합니다.
- 합이 3을 초과한 더미는 미획득 카드로 제거하고 초기화합니다.


In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'card_appearance_probability_100games.py').exists():
    NOTEBOOK_DIR = Path(r'C:\\sk-encoa\\codex_game\\programer\\probability')

SCRIPT = NOTEBOOK_DIR / 'card_appearance_probability_100games.py'
PYTHON = Path(r'C:\\sk-encoa\\data-collection-workspace\\.venv\\Scripts\\python.exe')
OUTPUT_DIR = NOTEBOOK_DIR / 'output'
GAMES = 100
SEED = 20260806

print('script =', SCRIPT)
print('output =', OUTPUT_DIR)


In [ ]:
command = [
    str(PYTHON), str(SCRIPT),
    '--games', str(GAMES),
    '--seed', str(SEED),
    '--output-dir', str(OUTPUT_DIR),
]
completed = subprocess.run(command, check=True, text=True, capture_output=True)
print(completed.stdout)


In [ ]:
card_df = pd.read_csv(OUTPUT_DIR / 'card_appearance_100_games.csv')
game_df = pd.read_csv(OUTPUT_DIR / 'game_summary_100_games.csv')
event_df = pd.read_csv(OUTPUT_DIR / 'removal_events_100_games.csv')
skull_df = pd.read_csv(OUTPUT_DIR / 'appearance_by_skull_100_games.csv')

print('카드별 등장 횟수 상위 15장')
display(card_df.head(15))
print('카드별 등장 횟수 하위 15장')
display(card_df.tail(15).sort_values('appearance_count'))


In [ ]:
print('100게임 요약')
display(game_df.describe(include='all'))

print('승자 분포')
display(game_df['winner'].value_counts(dropna=False).rename_axis('winner').reset_index(name='games'))

print('선택 더미에서 사라진 카드 수별 벨 이벤트')
display(event_df['acquired_card_count'].value_counts().sort_index().rename_axis('card_count').reset_index(name='events'))


In [ ]:
print('해골 수별 등장·획득 결과')
display(skull_df)

mean_count = card_df['appearance_count'].mean()
spread = card_df['appearance_count'].max() - card_df['appearance_count'].min()
print(f'카드당 평균 등장 횟수: {mean_count:.2f} / 100게임')
print(f'최다-최소 등장 횟수 차이: {spread:.0f}회')


## 결과 해석

- 덱의 각 위치에 배치될 사전 확률은 52장 모두 동일합니다.
- 게임이 3승에서 조기 종료되므로 모든 카드가 매 게임 등장하는 것은 아닙니다.
- 해골 값이 벨 발생과 종료 시점을 바꾸므로 해골 그룹별 등장·획득률 차이가 생길 수 있습니다.
- 100게임에서 카드별 차이가 보여도 표본 오차일 수 있으므로 실제 밸런스 확정 전 반복 수를 늘려 재검증해야 합니다.
